# 12 — Observability:LangSmith 追蹤與評估

**這份要學什麼**
- LangSmith 怎麼追蹤（trace）、評估（evaluate）agent 的行為
- LangSmith 跟開源自架的 Langfuse 差在哪裡、怎麼選

> **這份 notebook 跟前面幾份不太一樣。** 前面每份都能完全離線、確定性地跑出真正的結果,
> 因為我們用 `scripted_model` 取代了真的 LLM。但 LangSmith 的核心價值——追蹤面板、trace
> 視覺化、dataset/evaluator——是一個雲端服務,沒有本地離線的等價物。這份 notebook 的程式碼
> 一樣不需要 key 就能讀懂、能執行(不會報錯),但要真的「看到」trace 畫面,需要一個免費的
> [LangSmith 帳號](https://smith.langchain.com)——這裡沒辦法假裝儀表板畫面能在這裡直接
> 看到。

**這章在解決什麼問題**:agent 出錯或變慢時,你怎麼知道是哪一步出狀況?**LangSmith 就像
幫你的 agent 裝一台行車紀錄器/監視器**——每一步做了什麼、呼叫了哪個工具、花了多久、
LLM 實際回了什麼,全部錄下來。出事的時候回放紀錄片找原因,而不是憑猜測除錯。

## 為什麼要補這一塊
回顧 LangChain Academy 的課程列表時發現:官方課程有一大塊都是 LangSmith(observability、
evaluation、deployment、monitoring),但我們前 11 份 notebook 完全沒碰這個主題——
`10_langgraph_persistence_deploy.ipynb` 裡標記「部署概念是概念性、未跟官方文件核對」的
段落,其實有一部分就是 LangSmith Deployment 在做的事。這份 notebook 補上這個缺口的基礎
概念。

In [1]:
import sys

sys.path.insert(0, ".")

# 一定要先 import _langsmith（它會 load_dotenv()），再 import 任何 langchain / langgraph
# 模組——原因在下面「陷阱」那一段解釋。
from _langsmith import has_langsmith_key

print("LANGSMITH_API_KEY 已設定:", has_langsmith_key())

LANGSMITH_API_KEY 已設定: False


## 核心概念:跟你已經看過的東西對照

LangSmith 的資料模型,其實跟你在 `08_langgraph_streaming.ipynb` 玩過的
`stream_mode="debug"` 很像,只是規模做大、做成永久保存、加上一個網頁 UI。

一次完整的對話錄下來大概長這樣(這是示意圖,不是這份 notebook 實際跑出來的畫面):

```
使用者發問:「幫我查一下今天天氣」
        │
        ▼
┌───────────────────── Trace(這一次 invoke 的完整記錄) ─────────────────────┐
│                                                                             │
│  Run: node "agent"                                                         │
│    └─ Run: 呼叫 LLM   (耗時 0.8s,輸入/輸出全部存下來)                       │
│  Run: node "tools"                                                         │
│    └─ Run: 呼叫「查天氣」工具 (輸入參數、回傳結果全部存下來)                  │
│  Run: node "agent"(再問一次 LLM,決定要不要繼續)                            │
│    └─ Run: 呼叫 LLM                                                        │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
        │
        ▼
   回覆使用者「今天晴天,28度」
```

對照表:

| LangSmith 概念 | 白話解釋 | 對照我們學過的東西 |
|---|---|---|
| **Project** | 一組 trace 放在一起的容器 | 通常對應一個應用 / 環境(dev、staging、prod) |
| **Trace** | 一次完整的錄影,從頭到尾 | 一次完整的 `.invoke()` 呼叫 |
| **Run** | 錄影裡的一個片段/鏡頭 | 跟 `stream_mode="debug"` 印出的每個 `task` 事件是同一層級,差別是 Run 會被永久存下來、能在 UI 裡展開看輸入輸出、耗時、token 用量 |
| **Feedback** | 幫某個 Run 打分數、貼標籤 | 人工標註或自動評分都算 |
| **Dataset** | 一組「輸入 → 預期輸出」的題庫 | 拿來跑實驗、比較不同 prompt / 模型版本 |
| **Evaluator** | 自動改考卷的閱卷老師 | 對照 Dataset 的標準答案自動打分(LLM-as-judge、規則式、或自訂函式) |

## 陷阱:環境變數要在任何 LangChain import 之前就設定好

`langsmith` 套件用 `functools.lru_cache` 快取「有沒有開啟 tracing」這個檢查結果——**同一個
process 裡,只要檢查過一次,之後就算你再改 `os.environ`,結果也不會變。**

比喻:這就像球隊要交「教練名單」——名單要在比賽開始前、球員入場前就交出去。球賽開打之後
才想交名單,裁判不會理你,場上球員也不會變。`tracing_is_enabled()` 就是那份「名單」,一旦
被檢查過一次(等於「比賽開始」),之後再改環境變數也沒用。

下面直接示範這個行為(純粹測試快取機制,不需要真的有 key,也不會發送任何請求)。

In [2]:
import os

from langsmith.utils import tracing_is_enabled

print("第一次檢查:", tracing_is_enabled())

os.environ["LANGSMITH_TRACING"] = "true"
print("設定環境變數之後再檢查:", tracing_is_enabled(), "<- 沒變，因為結果已經被快取住了")

第一次檢查: False
設定環境變數之後再檢查: False <- 沒變，因為結果已經被快取住了


**實務上的結論**:不要在 notebook 中途才用 `os.environ["LANGSMITH_TRACING"] = ...` 開啟
tracing,那可能已經「開打之後才交名單」,太晚了。正確做法是寫進 `.env`,並確保
`load_dotenv()`(我們的 `_langsmith.py` 在 import 時就會呼叫)是這個 kernel 裡**第一件**
發生的事——這也是為什麼最上面的 import 順序特別註明「先 import `_langsmith`,再 import
其他東西」。

## 啟用 tracing:只要設環境變數,程式碼不用改

把這三行寫進 `.env`(複製 `.env.example` 並填入):

```bash
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=你的_key
LANGSMITH_PROJECT=learn-langgraph
```

接著**任何**用 LangChain / LangGraph 呼叫模型或跑 graph 的程式碼,都會自動把 trace 送到
LangSmith,不需要改一行 agent 的程式碼——因為 tracing 是透過 LangChain 的 callback 系統
全域接上去的,跟你在 `02`～`10` 寫的 `create_agent` / `StateGraph` 完全無關,純粹是「監聽
每個 Runnable 的呼叫」。下面這個 cell 會檢查兩個 key 是否都設定好,沒設定就跳過真的呼叫,
只印出說明文字。

In [3]:
from _llm import get_llm, has_api_key

if has_langsmith_key() and has_api_key():
    from langchain.agents import create_agent

    agent = create_agent(model=get_llm(), tools=[])
    agent.invoke({"messages": [("human", "用一句話介紹 LangSmith")]})

    project = os.environ.get("LANGSMITH_PROJECT", "default")
    print(f"呼叫完成，去 https://smith.langchain.com 的 '{project}' project 底下應該能看到這筆 trace。")
else:
    print("尚未設定 LANGSMITH_API_KEY 和/或 OPENAI_API_KEY，跳過真的呼叫。")
    print("概念上：只要上面兩個 key 都設定好，這個 cell 什麼都不用改就會自動送出 trace。")

尚未設定 LANGSMITH_API_KEY 和/或 OPENAI_API_KEY，跳過真的呼叫。
概念上：只要上面兩個 key 都設定好，這個 cell 什麼都不用改就會自動送出 trace。


## `@traceable`:追蹤任何 Python 函式,不限於 LangChain 呼叫

如果你的 pipeline 裡有一段自己寫的前處理/後處理邏輯(不是 LangChain 的 `Runnable`),想讓
它也出現在同一棵 trace tree 裡,包上 `@traceable` decorator 就好。**這個 decorator 在沒有
設定 key 時是安全的 no-op**——不會報錯,只是不會真的送資料出去,所以可以直接執行看看效果。

In [4]:
from langsmith import traceable


@traceable
def normalize_city_name(raw: str) -> str:
    """A plain Python preprocessing step -- not a LangChain Runnable."""
    return raw.strip().title()


print(normalize_city_name("  taipei "))

Taipei


## Dataset + Evaluator(概念)

這部分需要一個真的帳號才能操作,這裡先建立心智模型,之後真的要導入評估流程時再深入:

1. 準備一組 **Dataset**:每筆是 `{"inputs": {...}, "outputs": {...}}`(輸入 + 預期輸出),
   像一份「考古題+標準答案」
2. 寫一個 **Evaluator** 函式:扮演閱卷老師,拿到「模型的實際輸出」跟「Dataset 裡的標準
   答案」,回傳一個分數
3. 呼叫 `langsmith.evaluate(target_fn, data=dataset_name, evaluators=[...])`,LangSmith
   會自動跑過整份考古題、記錄每一題的分數,UI 上可以比較不同版本(換 prompt / 換模型)的
   分數變化

這跟 `07_langgraph_human_in_the_loop.ipynb` 手動核准草稿的邏輯是同一個精神——只是打分數
的人從「人」換成「自動化的 evaluator」,而且是一整批一起跑,不是一筆一筆審。

## LangSmith vs Langfuse：怎麼選

| | LangSmith | Langfuse |
|---|---|---|
| **授權** | 閉源商業產品 | 開源（可自架），雲端版另有付費方案 |
| **自架** | Enterprise 方案才能自架 | 免費就能用 Docker / Kubernetes 自架 |
| **跟 LangChain/LangGraph 的耦合** | 官方親兒子，callback 整合最順、新功能第一手支援 | 框架無關，靠 SDK / OpenTelemetry / 100+ 整合，含 LangChain 整合但非官方 |
| **免費額度（雲端版）** | 5,000 base traces/月，1 席 | 50,000 units/月，2 席 |
| **核心功能** | Tracing、Evaluation、Prompt Engineering、Deployment | Tracing、Prompt 管理、Evaluation、Monitoring |
| **適合情境** | 已經全套用 LangChain/LangGraph、想要跟框架整合最緊密、不排斥資料留在對方雲端 | 想要開源 / 資料自己留著（合規要求）、應用不是純 LangChain 生態、或想要跨框架統一觀測 |

**選型建議**：如果你的 agent 就是用 LangChain / LangGraph 建的，LangSmith 的整合摩擦力
最低（環境變數設好就自動追蹤，跟這份 notebook示範的一樣）。如果組織對「資料不能離開自己
環境」有硬性要求、或者你的技術棧本來就混合多種框架（不只 LangChain），開源自架的 Langfuse
會更適合——概念模型（trace / observation / dataset / evaluator）幾乎是共通的，選哪個
主要是「授權 / 自架 / 跟 LangChain 的整合深度」的取捨，不是功能有本質差異。

## 小結
- LangSmith 的核心價值是雲端追蹤面板(監視器/行車紀錄器),這是這門課裡唯一沒辦法完全
  離線體驗的部分
- 開啟 tracing 只需要環境變數,agent 的程式碼完全不用改;但**環境變數要在任何 LangChain
  import 之前就設定好**,不然會撞到 `lru_cache` 的「教練名單」陷阱——比賽開打後才交名單,
  系統不會理你
- `@traceable` 讓非 LangChain 的 plain function 也能進到同一棵 trace tree,而且沒 key
  時是安全的 no-op
- Langfuse 是開源、可自架的替代方案,概念模型跟 LangSmith 幾乎共通,選擇主要看授權/自架
  需求,而不是功能落差

到這裡,12 份 notebook 涵蓋了從 LCEL 到 LangGraph 核心機制、再到觀測與評估的完整入門路徑。